# Tutorial 04: QEC-Aware Compilation (Preview)

Quantum Error Correction (QEC) is essential for fault-tolerant quantum computing. qb-compiler's Phase 2 roadmap includes passes that are *QEC-aware* -- they understand the structure of error-correcting codes and compile circuits accordingly.

This tutorial previews these concepts:

1. Why QEC matters for compilation
2. The `qec_aware` flag in `QBCompiler.compile()`
3. Understanding surface code basics and logical qubits
4. How calibration data feeds into QEC decisions
5. What's coming: logical mapping, syndrome scheduling, and correlated error avoidance

**Note:** QEC-aware passes are under active development. This tutorial uses the existing API to show the foundation and previews the planned interfaces.

## 1. Why QEC Matters for Compilation

Current quantum hardware has gate errors around 0.1-1%. For useful algorithms (Shor's, chemistry simulations, optimization), we need effective error rates of 10^-10 or better. QEC bridges this gap by encoding logical qubits into multiple physical qubits.

The challenge: QEC dramatically increases the number of physical qubits and operations needed. A compiler that understands QEC can:

- **Map logical qubits** to patches of physical qubits that match the error-correcting code's geometry
- **Schedule syndrome extraction** between logical gates to catch errors early
- **Avoid correlated errors** that break the independence assumption QEC relies on
- **Minimize overhead** by choosing the right code distance based on the circuit's depth and the hardware's error rates

## 2. The qec_aware Flag

The `compile()` method already accepts a `qec_aware` parameter as a forward-looking flag. In Phase 1, this is a no-op, but it establishes the interface.

In [ ]:
from qb_compiler import QBCompiler, QBCircuit

In [ ]:
# Build a simple circuit
circ = QBCircuit(3).h(0).cx(0, 1).cx(1, 2).measure_all()

compiler = QBCompiler.from_backend("ibm_fez")

# Standard compilation
result_standard = compiler.compile(circ)
print(f"Standard: depth={result_standard.compiled_depth}, fidelity={result_standard.estimated_fidelity:.4f}")

# QEC-aware compilation (currently same output, reserved for Phase 2)
result_qec = compiler.compile(circ, qec_aware=True)
print(f"QEC-aware: depth={result_qec.compiled_depth}, fidelity={result_qec.estimated_fidelity:.4f}")

print("\nNote: In Phase 1, qec_aware=True produces the same output.")
print("Phase 2 will add QEC-specific passes when this flag is set.")

## 3. Surface Code Concepts

The surface code is the leading QEC code for superconducting qubits because it requires only nearest-neighbor connectivity (which matches hardware like IBM's heavy-hex lattice).

Key concepts:

- **Data qubits** store the logical quantum information
- **Ancilla qubits** are used for syndrome extraction (measuring error syndromes without disturbing data)
- **Code distance *d*** determines protection level: can correct up to floor((d-1)/2) errors
- A distance-*d* surface code patch uses *d x d* data qubits + *(d-1) x (d-1)* ancilla qubits

Let's explore how physical qubit requirements scale.

In [ ]:
def surface_code_physical_qubits(distance: int) -> int:
    """Number of physical qubits for one logical qubit at code distance d.

    A distance-d rotated surface code uses d^2 data qubits
    and (d^2 - 1) ancilla qubits.
    """
    data = distance ** 2
    ancilla = distance ** 2 - 1
    return data + ancilla


def logical_error_rate(physical_error: float, distance: int) -> float:
    """Rough estimate of logical error rate per QEC round.

    Uses the approximation: p_L ~ (p / p_threshold)^((d+1)/2)
    where p_threshold ~ 1% for surface codes.
    """
    p_threshold = 0.01
    if physical_error >= p_threshold:
        return 1.0  # below threshold, QEC doesn't help
    return (physical_error / p_threshold) ** ((distance + 1) / 2)


print(f"{'Distance':>10s}  {'Phys Qubits':>12s}  {'Correctable':>12s}  {'Logical err (p=0.1%)':>22s}  {'Logical err (p=0.5%)':>22s}")
print("-" * 85)

for d in [3, 5, 7, 9, 11, 13, 15]:
    phys = surface_code_physical_qubits(d)
    correctable = (d - 1) // 2
    le_01 = logical_error_rate(0.001, d)
    le_05 = logical_error_rate(0.005, d)
    print(
        f"{d:>10d}  "
        f"{phys:>12d}  "
        f"{correctable:>12d}  "
        f"{le_01:>22.2e}  "
        f"{le_05:>22.2e}"
    )

In [ ]:
# How many logical qubits can different backends support?
from qb_compiler import BACKEND_CONFIGS

print("Logical qubit capacity at different code distances:")
print(f"{'Backend':20s}  {'Phys Qubits':>12s}  {'d=3':>6s}  {'d=5':>6s}  {'d=7':>6s}  {'d=9':>6s}")
print("-" * 65)

for name, spec in BACKEND_CONFIGS.items():
    caps = []
    for d in [3, 5, 7, 9]:
        phys_per_logical = surface_code_physical_qubits(d)
        n_logical = spec.n_qubits // phys_per_logical
        caps.append(n_logical)
    print(
        f"{name:20s}  {spec.n_qubits:>12d}  "
        f"{caps[0]:>6d}  {caps[1]:>6d}  {caps[2]:>6d}  {caps[3]:>6d}"
    )

## 4. How Calibration Data Feeds QEC

QEC performance is highly sensitive to the physical error rates of the qubits and gates used. The same calibration data that powers the `CalibrationMapper` will be essential for QEC-aware compilation.

Key questions the QEC-aware compiler will answer:
- Which code distance is needed for this circuit on this hardware?
- Which physical qubits should form each logical qubit patch?
- Are there correlated errors that violate the independence assumption?

In [ ]:
# Use calibration data to determine the minimum code distance needed

def recommend_distance(
    physical_error: float,
    target_logical_error: float,
    max_distance: int = 25,
) -> int | None:
    """Find the minimum code distance to achieve a target logical error rate."""
    for d in range(3, max_distance + 1, 2):  # odd distances only
        le = logical_error_rate(physical_error, d)
        if le <= target_logical_error:
            return d
    return None  # can't achieve target


print("Minimum code distance for different backends and targets:")
print(f"{'Backend':20s}  {'Median CX err':>14s}  {'d for 1e-6':>12s}  {'d for 1e-10':>12s}  {'d for 1e-15':>12s}")
print("-" * 75)

for name, spec in BACKEND_CONFIGS.items():
    p = spec.median_cx_error
    d6 = recommend_distance(p, 1e-6)
    d10 = recommend_distance(p, 1e-10)
    d15 = recommend_distance(p, 1e-15)
    print(
        f"{name:20s}  {p:>14.4f}  "
        f"{str(d6) if d6 else 'N/A':>12s}  "
        f"{str(d10) if d10 else 'N/A':>12s}  "
        f"{str(d15) if d15 else 'N/A':>12s}"
    )

In [ ]:
# For each backend, what's the total physical qubit cost of a 10-logical-qubit circuit?

target_logical_qubits = 10
target_logical_error = 1e-10

print(f"\nPhysical qubit cost for {target_logical_qubits} logical qubits at p_L <= {target_logical_error}:")
print(f"{'Backend':20s}  {'Phys err':>10s}  {'Distance':>10s}  {'Phys/logical':>14s}  {'Total phys':>12s}  {'Available':>10s}  {'Feasible':>10s}")
print("-" * 95)

for name, spec in BACKEND_CONFIGS.items():
    d = recommend_distance(spec.median_cx_error, target_logical_error)
    if d:
        per_logical = surface_code_physical_qubits(d)
        total = per_logical * target_logical_qubits
        feasible = "YES" if total <= spec.n_qubits else "no"
        print(
            f"{name:20s}  {spec.median_cx_error:>10.4f}  {d:>10d}  "
            f"{per_logical:>14d}  {total:>12d}  {spec.n_qubits:>10d}  {feasible:>10s}"
        )
    else:
        print(f"{name:20s}  {spec.median_cx_error:>10.4f}  {'N/A':>10s}  below threshold")

## 5. What's Coming in Phase 2

qb-compiler's `passes/qec/` directory contains the planned QEC-aware passes:

### 5.1 Logical Mapping (`logical_mapping.py`)

Maps each logical qubit in a QEC circuit to a *patch* of physical qubits that forms a surface code tile. Uses calibration data to select the best patches -- avoiding noisy qubits that would increase the effective code distance needed.

### 5.2 Syndrome Scheduling (`syndrome_scheduling.py`)

Interleaves syndrome extraction rounds with logical gate operations. The scheduler will:
- Insert syndrome measurements at optimal intervals based on decoherence rates
- Minimise the latency between syndrome rounds to catch errors before they propagate
- Account for measurement and reset times from calibration data

### 5.3 Correlated Error Avoidance (`correlated_error_avoidance.py`)

This is unique to qb-compiler. QubitBoost's SafetyGate system detects temporal correlations in hardware errors. When errors on neighbouring qubits are correlated (violating the i.i.d. assumption), the surface code's effective distance is reduced.

The correlated error avoidance pass will:
- Query SafetyGate metrics for temporal correlation scores
- Remap logical patches to avoid physically adjacent qubits with correlated errors
- Adjust code distance when correlations are detected

In [ ]:
# Preview: what the QEC pass pipeline will look like

planned_qec_passes = [
    "DepthAnalysis",                   # Analyze circuit depth
    "CalibrationMapper",                # Calibration-aware initial mapping
    "LogicalQubitMapper",               # Map logical qubits to surface code patches  [Phase 2]
    "SyndromeScheduler",                # Insert syndrome extraction rounds            [Phase 2]
    "CorrelatedErrorAvoidance",         # Avoid correlated error regions                [Phase 2]
    "NoiseAwareRouter",                 # Route using calibration data
    "GateDecomposition",                # Decompose to native basis
    "GateCancellation",                 # Cancel redundant gates
    "NoiseAwareScheduler",              # Schedule around T1/T2 decay
]

print("Planned QEC-aware pass pipeline:")
for i, name in enumerate(planned_qec_passes, 1):
    phase = "[Phase 2]" if any(x in name for x in ["Logical", "Syndrome", "Correlated"]) else "[Phase 1]"
    print(f"  {i:2d}. {name:35s}  {phase}")

## 6. Preparing for QEC Today

Even without full QEC support, you can use qb-compiler's existing tools to plan QEC experiments.

In [ ]:
# Use the fidelity estimator to determine when QEC becomes necessary
from qb_compiler.noise.fidelity_estimator import FidelityEstimator

estimator = FidelityEstimator()

# At what circuit depth does fidelity drop below useful thresholds?
print("Circuit depth vs. estimated fidelity (IBM Fez median errors):")
print(f"{'Depth':>8s}  {'2Q gates':>10s}  {'Est. Fidelity':>14s}  {'Useful?':>10s}")
print("-" * 48)

for depth in [5, 10, 20, 50, 100, 200, 500, 1000]:
    # Rough approximation: about 1 CX per depth layer
    n_cx = depth
    n_1q = depth * 2
    f = estimator.estimate_depth_limited_fidelity(
        n_two_qubit_gates=n_cx,
        avg_two_qubit_error=0.005,
        n_one_qubit_gates=n_1q,
        avg_one_qubit_error=0.0005,
    )
    useful = "yes" if f > 0.5 else ("marginal" if f > 0.1 else "NO")
    print(f"{depth:>8d}  {n_cx:>10d}  {f:>14.4f}  {useful:>10s}")

print("\nCircuits beyond ~100 CX gates on IBM Fez need QEC to be useful.")

In [ ]:
# Cost analysis: what would a QEC-protected experiment cost?
from qb_compiler.cost.pricing import cost_per_shot

print("Cost of QEC-protected circuits (1000 shots):")
print(f"{'Backend':20s}  {'Distance':>10s}  {'Phys qubits':>12s}  {'Est. cost':>12s}")
print("-" * 60)

n_logical = 5  # 5 logical qubits
target_shots = 1000

for name in ["ibm_fez", "ibm_torino", "ionq_aria"]:
    spec = BACKEND_CONFIGS[name]
    d = recommend_distance(spec.median_cx_error, 1e-8)
    if d:
        phys = surface_code_physical_qubits(d) * n_logical
        if phys <= spec.n_qubits:
            cps = cost_per_shot(name)
            # QEC circuits are much deeper: assume ~10x depth overhead
            depth_factor = max(1.0, (d * 10) / 100.0)
            total = cps * depth_factor * target_shots
            print(f"{name:20s}  {d:>10d}  {phys:>12d}  ${total:>10.2f}")
        else:
            print(f"{name:20s}  {d:>10d}  {phys:>12d}  not enough qubits")
    else:
        print(f"{name:20s}  {'N/A':>10s}  below threshold")

## Summary

In this tutorial you learned:

- **Why QEC matters for compilation** -- error rates must drop from ~0.1% to ~10^-10 for useful algorithms
- **Surface code basics** -- data qubits, ancilla qubits, code distance, and the qubit overhead
- **The qec_aware flag** -- already available in `compile()`, reserved for Phase 2 QEC-specific passes
- **Calibration meets QEC** -- hardware error rates determine the required code distance and qubit selection
- **Phase 2 roadmap** -- logical mapping, syndrome scheduling, and correlated error avoidance
- **Planning tools** -- use `FidelityEstimator` and cost analysis to determine when QEC is needed and what it will cost

### Getting Involved

QEC-aware compilation is under active development. If you're interested in contributing:

- Check the `src/qb_compiler/passes/qec/` directory for the pass stubs
- See `CONTRIBUTING.md` for development setup
- File issues or feature requests at the project repository